In [1]:
# ============================================================
# SUBSET EXPERIMENT: EVALUATING INDUCTIVE BIAS HYPOTHESIS
#
# Models: Baseline vs  Latent CRF
# Backbone: 3 hidden layers (optimal from depth experiment)
# Training data subsets: 100%, 80%, 60%, 40%, 20%, 10%
# Validation and Test sets: FULL (unchanged)
# ============================================================


# ============================================================
# CELL 1: IMPORTS, SEED, DEVICE
# ============================================================

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, hamming_loss, accuracy_score, roc_curve
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import random
import os
import json
import shutil
from tqdm.notebook import tqdm
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")



Using device: cpu


In [2]:
# ============================================================
# CELL 2: PREPROCESSING AND DATA LOADING
# ============================================================

df = pd.read_csv('data.csv')
print(f"Loaded {len(df)} rows")

def clean_dia_life(x):
    if pd.isna(x):
        return np.nan
    x_str = str(x).lower().strip()
    if 'month' in x_str or x_str.endswith('m'):
        num = ''.join(filter(lambda c: c.isdigit() or c == '.', x_str))
        try:
            return float(num) / 12
        except:
            return np.nan
    else:
        try:
            return float(x_str)
        except:
            return np.nan

df['DIA LIFE'] = df['DIA LIFE'].apply(clean_dia_life)

complications = ['NEP', 'NEU', 'RET']
exclude_cols  = ['SL.NO', 'NAME'] + complications
feature_cols  = [c for c in df.columns if c not in exclude_cols]
print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

df_clean = df.dropna(subset=feature_cols + complications)
print(f"After dropping missing: {len(df_clean)} rows")

X = df_clean[feature_cols].values.astype(np.float32)
y = df_clean[complications].values.astype(np.float32)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

print("\nPositive counts:")
for i, comp in enumerate(complications):
    pos = (y[:, i] == 1).sum()
    print(f"  {comp}: {pos} ({pos/len(y)*100:.1f}%)")

y_combined = y.dot(2**np.arange(y.shape[1]))

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y_combined
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42,
    stratify=y_temp.dot(2**np.arange(y_temp.shape[1]))
)

print(f"\nSplit sizes:")
print(f"  Train: {len(X_train)}")
print(f"  Val:   {len(X_val)}")
print(f"  Test:  {len(X_test)}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

batch_size   = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size, shuffle=False)

input_dim = X_train.shape[1]

print(f"\nDataLoaders created:")
print(f"  Batch size:         {batch_size}")
print(f"  Training batches:   {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches:       {len(test_loader)}")
print(f"  Input dimension:    {input_dim}")

torch.save({
    'X_train': X_train_t, 'y_train': y_train_t,
    'X_val':   X_val_t,   'y_val':   y_val_t,
    'X_test':  X_test_t,  'y_test':  y_test_t,
    'scaler':        scaler,
    'feature_cols':  feature_cols,
    'complications': complications
}, 'processed_data.pt')

print("\n✅ Preprocessing complete.")



Loaded 3068 rows

Features (17): ['AGE', 'SEX', 'BMI', 'SP', 'BP', 'HbA1c', 'FPS', 'PPS', 'FAMILY H/O', 'ONSET AGE', 'DIA LIFE', 'SMOKING', 'PHY ACT', 'MED USE', 'MED ADH', 'CV', 'PER VAS']
After dropping missing: 3068 rows

X shape: (3068, 17)
y shape: (3068, 3)

Positive counts:
  NEP: 1468 (47.8%)
  NEU: 1492 (48.6%)
  RET: 1547 (50.4%)

Split sizes:
  Train: 2148
  Val:   459
  Test:  461

DataLoaders created:
  Batch size:         32
  Training batches:   68
  Validation batches: 15
  Test batches:       15
  Input dimension:    17

✅ Preprocessing complete.


In [3]:
# ============================================================
# CELL 3: MODEL DEFINITIONS (FIXED DEPTH = 3 HIDDEN LAYERS)
# ============================================================

class SharedBackbone(nn.Module):
    """
    Shared backbone — FIXED: hidden_dim=16, proj_dim=8, 3 hidden layers.
    Total params: ~1,107 | Samples/param: ~1.94
    """
    def __init__(self, input_dim, hidden_dim=16, n_hidden_layers=3, proj_dim=8):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers += [
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
            ]
            in_dim = hidden_dim
        layers += [
            nn.Linear(hidden_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        ]
        self.net        = nn.Sequential(*layers)
        self.output_dim = proj_dim

    def forward(self, x):
        return self.net(x)


class BaselineModel(nn.Module):
    """Shared backbone + independent linear head. No interaction."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head     = nn.Linear(backbone.output_dim, 3)

    def forward(self, x):
        return self.head(self.backbone(x))


class CRFHead(nn.Module):
    """
    CRF interaction head with mean-field inference (3 iterations).
    β is symmetric — β_ij = β_ji — enforced by using upper triangle only.
    Diagonal is zero by construction — no self-interaction.
    Assumes undirected co-occurrence: complications tend to appear together
    without directionality, consistent with shared microvascular pathway.
    """
    def __init__(self, hidden_dim, n_labels=3, n_iter=3, init_scale=0.01):
        super().__init__()
        self.base   = nn.Linear(hidden_dim, n_labels)
        self.n_iter = n_iter
        # Only upper triangle parameters — 3 values for K=3
        # β_01, β_02, β_12
        n_pairs = n_labels * (n_labels - 1) // 2
        self.beta_upper = nn.Parameter(torch.full((n_pairs,), init_scale))

    def _build_beta(self):
        """Reconstruct full symmetric beta matrix from upper triangle."""
        # For K=3: indices (0,1), (0,2), (1,2)
        K    = 3
        beta = torch.zeros(K, K, device=self.beta_upper.device)
        idx  = 0
        for i in range(K):
            for j in range(i+1, K):
                beta[i, j] = self.beta_upper[idx]
                beta[j, i] = self.beta_upper[idx]  # enforce symmetry
                idx += 1
        return beta  # diagonal remains zero

    def forward(self, h):
        base_logits = self.base(h)
        beta        = self._build_beta()
        # Mean-field iterations
        q = torch.sigmoid(base_logits)  # initialise at base probs
        for _ in range(self.n_iter):
            # q: (batch, K), beta: (K, K)
            # message: (batch, K) — weighted sum of other labels' beliefs
            message = q @ beta.T
            q       = torch.sigmoid(base_logits + message)
        # Return logits consistent with final q
        # Inverse sigmoid to get logits: log(q / (1-q))
        q       = q.clamp(1e-6, 1 - 1e-6)
        logits  = torch.log(q / (1 - q))
        return logits


class CRFModel(nn.Module):
    """Shared backbone + symmetric CRF interaction head."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head     = CRFHead(backbone.output_dim, n_labels=3)

    def forward(self, x):
        return self.head(self.backbone(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Verify parameter budget
bb       = SharedBackbone(input_dim)
base     = BaselineModel(bb)
bb_crf   = SharedBackbone(input_dim)
crf      = CRFModel(bb_crf)
n_params_base = count_params(base)
n_params_crf  = count_params(crf)
print(f"Architecture: hidden_dim=16, proj_dim=8, n_hidden_layers=3")
print(f"Baseline params:  {n_params_base:,}")
print(f"CRF params:       {n_params_crf:,}  "
      f"(+{n_params_crf - n_params_base} from beta_upper)")
print(f"Samples/param:    {len(X_train)/n_params_crf:.2f}")
print("\n✅ Model definitions ready (CRF, symmetric, 3 mean-field iterations).")

Architecture: hidden_dim=16, proj_dim=8, n_hidden_layers=3
Baseline params:  1,107
CRF params:       1,110  (+3 from beta_upper)
Samples/param:    1.94

✅ Model definitions ready (CRF, symmetric, 3 mean-field iterations).


In [4]:

# ============================================================
# CELL 4: TRAINING FUNCTION (WITH TQDM)
# ============================================================

def train_model(model, train_loader, val_loader, model_name,
                epochs=300, lr=0.001, patience=40):
    model     = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=15, factor=0.5
    )

    train_losses     = []
    val_aurocs       = []
    best_val_auroc   = 0
    patience_counter = 0
    best_state       = None

    pbar = tqdm(range(epochs), desc=f'{model_name}', unit='epoch',
                bar_format='{l_bar}{bar:30}{r_bar}')

    for epoch in pbar:
        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        # ── Validate ──────────────────────────────────────────────────────
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                all_logits.append(model(X_batch.to(device)).cpu())
                all_labels.append(y_batch.cpu())

        all_probs  = torch.sigmoid(torch.cat(all_logits))
        all_labels = torch.cat(all_labels)
        val_auroc  = roc_auc_score(all_labels.numpy(), all_probs.numpy(), average='macro')
        val_aurocs.append(val_auroc)

        scheduler.step(val_auroc)
        lr_now = optimizer.param_groups[0]['lr']

        pbar.set_postfix({
            'loss':      f'{epoch_loss:.4f}',
            'val_auroc': f'{val_auroc:.4f}',
            'best':      f'{best_val_auroc:.4f}',
            'lr':        f'{lr_now:.6f}',
            'patience':  f'{patience_counter}/{patience}'
        })

        if val_auroc > best_val_auroc:
            best_val_auroc   = val_auroc
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                pbar.set_description(f'{model_name} [EARLY STOP ep={epoch+1}]')
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_aurocs, best_val_auroc


print("✅ Training function ready.")


✅ Training function ready.


In [5]:
# ============================================================
# CELL 5: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, test_loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            all_logits.append(model(X_batch.to(device)).cpu())
            all_labels.append(y_batch)

    probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    labels = torch.cat(all_labels).numpy()
    preds  = (probs > 0.5).astype(int)

    per_label_auroc = {
        comp: roc_auc_score(labels[:, i], probs[:, i])
        for i, comp in enumerate(complications)
    }

    return {
        'auroc_macro':      roc_auc_score(labels, probs, average='macro'),
        'per_label_auroc':  per_label_auroc,
        'f1_macro':         f1_score(labels, preds, average='macro'),
        'hamming_loss':     hamming_loss(labels, preds),
        'subset_accuracy':  accuracy_score(labels, preds),
        'probabilities':    probs,
        'labels':           labels,
    }


print("✅ Evaluation function ready.")


✅ Evaluation function ready.


In [6]:
# ============================================================
# CELL 6: PLOTTING FUNCTION
# ============================================================

COMP_FULL = ['Nephropathy', 'Neuropathy', 'Retinopathy']
C_BASE    = '#2563EB'
C_LIN     = '#DC2626'

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.color':        '#E5E7EB',
    'grid.linewidth':    0.6,
    'axes.labelsize':    11,
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'legend.framealpha': 0.9,
    'figure.dpi':        150,
})


def save_all_plots(data_percent, SAVE_DIR,
                   baseline_losses, linear_losses,
                   baseline_aurocs, linear_aurocs,
                   baseline_best_val, linear_best_val,
                   baseline_results, linear_results,
                   A_matrix):

    plt.close('all')
    percent_str = f'{int(data_percent*100)}% Data'

    # ── PLOT 1: TRAINING DYNAMICS ─────────────────────────────────────────
    try:
        plt.close('all')
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'Training Dynamics — {percent_str}',
                     fontsize=14, fontweight='bold', y=1.01)

        ax = axes[0]
        ax.plot(baseline_losses, color=C_BASE, linewidth=2, label='Baseline')
        ax.plot(linear_losses,   color=C_LIN,  linewidth=2, label='Interaction Model')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Training Loss (BCE)')
        ax.set_title('Training Loss')
        ax.legend()

        ax = axes[1]
        ax.plot(baseline_aurocs, color=C_BASE, linewidth=2,
                label=f'Baseline (best = {baseline_best_val:.4f})')
        ax.plot(linear_aurocs,   color=C_LIN,  linewidth=2,
                label=f'Interaction Model (best = {linear_best_val:.4f})')
        ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1,
                   alpha=0.5, label='Random baseline (0.5)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation AUROC (macro)')
        ax.set_title('Validation AUROC')
        ax.set_ylim(bottom=0.45)
        ax.legend()

        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, 'plot1_training_dynamics.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✅ Plot 1: Training dynamics")
    except Exception as e:
        plt.close('all')
        print(f"  ⚠️  Plot 1 skipped: {e}")

    # ── PLOT 2: METRIC COMPARISON ─────────────────────────────────────────
    try:
        metrics = {
            'AUROC\n(macro)':    (baseline_results['auroc_macro'],     linear_results['auroc_macro']),
            'F1\n(macro)':       (baseline_results['f1_macro'],        linear_results['f1_macro']),
            'Subset\nAccuracy':  (baseline_results['subset_accuracy'], linear_results['subset_accuracy']),
            'Hamming\nLoss (↓)': (baseline_results['hamming_loss'],    linear_results['hamming_loss']),
        }

        plt.close('all')
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'Model Performance Comparison — {percent_str}',
                     fontsize=14, fontweight='bold', y=1.01)

        x           = np.arange(len(metrics))
        width       = 0.35
        labels_list = list(metrics.keys())
        base_vals   = [v[0] for v in metrics.values()]
        linear_vals = [v[1] for v in metrics.values()]

        ax     = axes[0]
        bars_b = ax.bar(x - width/2, base_vals,   width, label='Baseline',         color=C_BASE, alpha=0.85)
        bars_l = ax.bar(x + width/2, linear_vals, width, label='Interaction Model', color=C_LIN,  alpha=0.85)
        for bar in bars_b:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f'{bar.get_height():.4f}', ha='center', va='bottom',
                    fontsize=8.5, color=C_BASE, fontweight='bold')
        for bar in bars_l:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f'{bar.get_height():.4f}', ha='center', va='bottom',
                    fontsize=8.5, color=C_LIN,  fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(labels_list)
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1.08)
        ax.legend()
        ax.set_title('All Metrics')

        ax     = axes[1]
        diffs  = [l - b for b, l in zip(base_vals, linear_vals)]
        colors = [C_LIN if d >= 0 else C_BASE for d in diffs]
        hamming_idx = labels_list.index('Hamming\nLoss (↓)')
        colors[hamming_idx] = C_LIN if diffs[hamming_idx] <= 0 else C_BASE
        bars = ax.bar(x, diffs, width=0.5, color=colors, alpha=0.85)
        ax.axhline(y=0, color='black', linewidth=0.8)
        for bar, d in zip(bars, diffs):
            ypos = bar.get_height() + 0.0005 if d >= 0 else bar.get_height() - 0.002
            ax.text(bar.get_x() + bar.get_width()/2, ypos,
                    f'{d:+.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(labels_list)
        ax.set_ylabel('Δ (Interaction Model − Baseline)')
        ax.set_title('Performance Gap')

        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, 'plot2_metric_comparison.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✅ Plot 2: Metric comparison")
    except Exception as e:
        plt.close('all')
        print(f"  ⚠️  Plot 2 skipped: {e}")

    # ── PLOT 3: PER-LABEL AUROC + ROC CURVES ─────────────────────────────
    try:
        plt.close('all')
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'Per-Label Analysis — {percent_str}',
                     fontsize=14, fontweight='bold', y=1.01)

        ax     = axes[0]
        base_p = [baseline_results['per_label_auroc'][c] for c in complications]
        lin_p  = [linear_results['per_label_auroc'][c]   for c in complications]
        x      = np.arange(3)
        width  = 0.35
        bars_b = ax.bar(x - width/2, base_p, width, label='Baseline',         color=C_BASE, alpha=0.85)
        bars_l = ax.bar(x + width/2, lin_p,  width, label='Interaction Model', color=C_LIN,  alpha=0.85)
        for bar in bars_b:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{bar.get_height():.4f}', ha='center', va='bottom',
                    fontsize=9, color=C_BASE, fontweight='bold')
        for bar in bars_l:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{bar.get_height():.4f}', ha='center', va='bottom',
                    fontsize=9, color=C_LIN,  fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(COMP_FULL)
        ax.set_ylabel('AUROC')
        ax.set_ylim(0.88, 1.02)
        ax.legend()
        ax.set_title('Per-Label AUROC')

        ax          = axes[1]
        base_probs  = baseline_results['probabilities']
        lin_probs   = linear_results['probabilities']
        true_labels = baseline_results['labels']
        linestyles  = ['-', '--', ':']
        for i, (comp, comp_full, ls) in enumerate(zip(complications, COMP_FULL, linestyles)):
            fpr_b, tpr_b, _ = roc_curve(true_labels[:, i], base_probs[:, i])
            fpr_l, tpr_l, _ = roc_curve(true_labels[:, i], lin_probs[:, i])
            auc_b = baseline_results['per_label_auroc'][comp]
            auc_l = linear_results['per_label_auroc'][comp]
            ax.plot(fpr_b, tpr_b, color=C_BASE, linestyle=ls, linewidth=1.8,
                    label=f'Base {comp_full[:3]} ({auc_b:.3f})')
            ax.plot(fpr_l, tpr_l, color=C_LIN,  linestyle=ls, linewidth=1.8,
                    label=f'Int  {comp_full[:3]} ({auc_l:.3f})')
        ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves (all labels)')
        ax.legend(fontsize=8.5, loc='lower right')

        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, 'plot3_per_label.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✅ Plot 3: Per-label AUROC + ROC curves")
    except Exception as e:
        plt.close('all')
        print(f"  ⚠️  Plot 3 skipped: {e}")

    # ── PLOT 4: INTERACTION MATRIX ────────────────────────────────────────
    try:
        plt.close('all')
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(f'Learned Interaction Matrix A — {percent_str}',
                     fontsize=14, fontweight='bold', y=1.01)

        ax     = axes[0]
        A_plot = np.clip(A_matrix, -10.0, 10.0)
        vmax   = float(np.clip(max(abs(A_plot.min()), abs(A_plot.max())) + 0.005, 1e-6, 10.0))
        norm   = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        im     = ax.imshow(A_plot, cmap='RdBu_r', norm=norm, aspect='auto')
        ax.set_xticks(range(3))
        ax.set_yticks(range(3))
        ax.set_xticklabels(COMP_FULL, rotation=20, ha='right')
        ax.set_yticklabels(COMP_FULL)
        ax.set_xlabel('Source (what does the influencing)', labelpad=8)
        ax.set_ylabel('Target (what gets influenced)',      labelpad=8)
        ax.set_title('A matrix heatmap')
        ax.grid(False)
        for i in range(3):
            for j in range(3):
                val   = float(A_plot[i, j])
                color = 'white' if abs(val) > vmax * 0.55 else 'black'
                ax.text(j, i, f'{val:+.4f}', ha='center', va='center',
                        fontsize=11, fontweight='bold', color=color)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Interaction weight', fontsize=10)

        ax         = axes[1]
        pairs      = []
        values     = []
        bar_colors = []
        for i, tgt in enumerate(COMP_FULL):
            for j, src in enumerate(COMP_FULL):
                if i != j:
                    pairs.append(f'{src[:3]}→{tgt[:3]}')
                    values.append(float(A_plot[i, j]))
                    bar_colors.append(C_LIN if A_plot[i, j] >= 0 else C_BASE)
        y_pos = np.arange(len(pairs))
        ax.barh(y_pos, values, color=bar_colors, alpha=0.85, height=0.6)
        ax.axvline(x=0, color='black', linewidth=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(pairs, fontsize=10)
        ax.set_xlabel('Interaction weight')
        ax.set_title('Off-diagonal interactions')
        for i, (val, yp) in enumerate(zip(values, y_pos)):
            xpos = val + 0.001 if val >= 0 else val - 0.001
            ha   = 'left'       if val >= 0 else 'right'
            ax.text(xpos, yp, f'{val:+.4f}', va='center', ha=ha,
                    fontsize=9, fontweight='bold')

        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, 'plot4_A_matrix.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✅ Plot 4: Interaction matrix")
    except Exception as e:
        plt.close('all')
        print(f"  ⚠️  Plot 4 skipped: {e}")


print("✅ Plotting function ready.")

✅ Plotting function ready.


In [7]:
# ============================================================
# CELL 7: MAIN SUBSET EXPERIMENT LOOP (MULTI-SEED)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

SEEDS            = [42, 123, 456, 789, 1337, 2024, 9999]
SUBSET_FRACTIONS = [1.0, 0.8, 0.6, 0.4, 0.2]

BASE_DIR  = 'results/subset_crf_v3/'
DRIVE_DIR = '/content/drive/MyDrive/comorbidity_experiments/subset_crf_v3/'
os.makedirs(BASE_DIR,  exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

all_results = {seed: {} for seed in SEEDS}

print(f"\n{'='*65}")
print("SUBSET EXPERIMENT — CRF INTERACTION (SYMMETRIC, 3 ITERATIONS)")
print(f"Seeds:     {SEEDS}")
print(f"Fractions: {SUBSET_FRACTIONS}")
print(f"Backbone:  hidden_dim=16, proj_dim=8, 3 hidden layers (fixed)")
print(f"Models:    Baseline vs CRF")
print(f"{'='*65}")

for seed in SEEDS:
    print(f"\n{'='*65}")
    print(f"SEED: {seed}")
    print(f"{'='*65}")

    seed_dir       = os.path.join(BASE_DIR,  f'seed_{seed}')
    seed_drive_dir = os.path.join(DRIVE_DIR, f'seed_{seed}')
    os.makedirs(seed_dir,       exist_ok=True)
    os.makedirs(seed_drive_dir, exist_ok=True)

    for fraction in SUBSET_FRACTIONS:
        percent_str = f'{int(fraction*100)}pct'
        print(f"\n{'='*60}")
        print(f"SEED {seed} — SUBSET CONDITION: {int(fraction*100)}% Data")
        print(f"{'='*60}")

        SAVE_DIR = os.path.join(seed_dir, percent_str)
        os.makedirs(SAVE_DIR, exist_ok=True)

        # ── Subsample training data ───────────────────────────────────────
        n_samples = len(X_train)
        n_subset  = int(n_samples * fraction)

        if fraction < 1.0:
            rng     = np.random.RandomState(seed)
            indices = rng.choice(n_samples, n_subset, replace=False)
            X_train_subset = X_train[indices]
            y_train_subset = y_train[indices]
            print(f"  Subsampled to {len(X_train_subset)} training samples ({int(fraction*100)}%)")
        else:
            X_train_subset = X_train
            y_train_subset = y_train
            print(f"  Using full training set: {len(X_train_subset)} samples")

        X_tr_t = torch.tensor(X_train_subset, dtype=torch.float32)
        y_tr_t = torch.tensor(y_train_subset, dtype=torch.float32)
        cond_train_loader = DataLoader(
            TensorDataset(X_tr_t, y_tr_t),
            batch_size=batch_size, shuffle=True
        )

        # ── Fresh backbone init for this seed ─────────────────────────────
        set_seed(seed)
        backbone   = SharedBackbone(input_dim)
        init_state = {k: v.cpu().clone() for k, v in backbone.state_dict().items()}

        torch.save(
            init_state,
            os.path.join(SAVE_DIR, 'backbone_init.pt')
        )

        # ── Train Baseline ────────────────────────────────────────────────
        set_seed(seed)
        baseline_model = BaselineModel(backbone)
        baseline_model, baseline_losses, baseline_aurocs, baseline_best_val = train_model(
            baseline_model, cond_train_loader, val_loader,
            f'Baseline | seed={seed}, {int(fraction*100)}% data'
        )
        baseline_results = evaluate_model(baseline_model, test_loader)
        print(f"Baseline Test AUROC: {baseline_results['auroc_macro']:.4f}")

        # ── Train CRF ─────────────────────────────────────────────────────
        backbone_crf = SharedBackbone(input_dim)
        backbone_crf.load_state_dict(init_state)

        set_seed(seed)
        crf_model = CRFModel(backbone_crf)
        crf_model, crf_losses, crf_aurocs, crf_best_val = train_model(
            crf_model, cond_train_loader, val_loader,
            f'CRF | seed={seed}, {int(fraction*100)}% data'
        )
        crf_results = evaluate_model(crf_model, test_loader)
        print(f"CRF Test AUROC: {crf_results['auroc_macro']:.4f}")

        # ── Extract beta matrix ───────────────────────────────────────────
        beta_matrix = crf_model.head._build_beta().detach().cpu().numpy()

        gap = crf_results['auroc_macro'] - baseline_results['auroc_macro']
        print(f"Gap: {gap:+.4f}")
        print(f"Beta (co-occurrence weights):")
        print(f"  NEP-NEU: {beta_matrix[0,1]:+.4f}")
        print(f"  NEP-RET: {beta_matrix[0,2]:+.4f}")
        print(f"  NEU-RET: {beta_matrix[1,2]:+.4f}")

        # ── Save plots ────────────────────────────────────────────────────
        try:
            save_all_plots(
                fraction, SAVE_DIR,
                baseline_losses, crf_losses,
                baseline_aurocs, crf_aurocs,
                baseline_best_val, crf_best_val,
                baseline_results, crf_results,
                beta_matrix
            )
        except Exception as e:
            print(f"  ⚠️ Plot error (non-fatal): {e}")

        # ── Save model weights ────────────────────────────────────────────
        torch.save(baseline_model.state_dict(),
                   os.path.join(SAVE_DIR, 'baseline_model.pt'))
        torch.save(crf_model.state_dict(),
                   os.path.join(SAVE_DIR, 'crf_model.pt'))
        np.save(os.path.join(SAVE_DIR, 'beta_matrix.npy'), beta_matrix)
        print("  ✅ Model weights and beta matrix saved")

        # ── Save results JSON ─────────────────────────────────────────────
        condition_results = {
            'seed':            seed,
            'data_fraction':   fraction,
            'n_train_samples': len(X_train_subset),
            'baseline': {
                'best_val_auroc':  baseline_best_val,
                'auroc_macro':     baseline_results['auroc_macro'],
                'per_label_auroc': baseline_results['per_label_auroc'],
                'f1_macro':        baseline_results['f1_macro'],
                'hamming_loss':    baseline_results['hamming_loss'],
                'subset_accuracy': baseline_results['subset_accuracy'],
                'train_losses':    baseline_losses,
                'val_aurocs':      baseline_aurocs,
            },
            'crf': {
                'best_val_auroc':  crf_best_val,
                'auroc_macro':     crf_results['auroc_macro'],
                'per_label_auroc': crf_results['per_label_auroc'],
                'f1_macro':        crf_results['f1_macro'],
                'hamming_loss':    crf_results['hamming_loss'],
                'subset_accuracy': crf_results['subset_accuracy'],
                'train_losses':    crf_losses,
                'val_aurocs':      crf_aurocs,
                'beta_matrix':     beta_matrix.tolist(),
            },
            'gap': gap,
        }

        with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
            json.dump(condition_results, f, indent=2)
        print("  ✅ Results JSON saved")

        # ── Copy to Drive ─────────────────────────────────────────────────
        drive_condition_dir = os.path.join(seed_drive_dir, percent_str)
        os.makedirs(drive_condition_dir, exist_ok=True)
        for filename in os.listdir(SAVE_DIR):
            shutil.copy2(
                os.path.join(SAVE_DIR, filename),
                os.path.join(drive_condition_dir, filename)
            )
        print(f"  ✅ Artifacts copied to Drive: {drive_condition_dir}")

        all_results[seed][fraction] = condition_results
        print(f"\n✅ COMPLETE: seed={seed}, {int(fraction*100)}% data")


# ============================================================
# MASTER SUMMARY
# ============================================================

print(f"\n{'='*70}")
print("SUBSET EXPERIMENT — CRF — MASTER SUMMARY")
print(f"{'='*70}")
print(f"\n{'Fraction':<12} {'Seed':<8} {'Base AUROC':<14} {'CRF AUROC':<16} {'Δ':<10} {'Winner'}")
print("-"*70)

for seed in SEEDS:
    for fraction in SUBSET_FRACTIONS:
        r      = all_results[seed][fraction]
        b_auc  = r['baseline']['auroc_macro']
        c_auc  = r['crf']['auroc_macro']
        gap    = r['gap']
        winner = 'CRF ✅' if gap > 0 else 'Baseline'
        print(f"{int(fraction*100):<12} {seed:<8} {b_auc:<14.4f} {c_auc:<16.4f} {gap:<+10.4f} {winner}")

print(f"\n{'='*70}")
print("AGGREGATED (mean ± std across seeds)")
print(f"{'='*70}")
print(f"\n{'Fraction':<12} {'Train N':<10} {'Base AUROC':<22} {'CRF AUROC':<22} {'Δ':<18} {'Winner'}")
print("-"*70)

for fraction in SUBSET_FRACTIONS:
    b_aucs = [all_results[s][fraction]['baseline']['auroc_macro'] for s in SEEDS]
    c_aucs = [all_results[s][fraction]['crf']['auroc_macro']      for s in SEEDS]
    gaps   = [all_results[s][fraction]['gap']                     for s in SEEDS]
    n_tr   = all_results[SEEDS[0]][fraction]['n_train_samples']
    winner = 'CRF ✅' if np.mean(gaps) > 0 else 'Baseline'
    print(f"{int(fraction*100):<12} {n_tr:<10} "
          f"{np.mean(b_aucs):.4f} ± {np.std(b_aucs):.4f}    "
          f"{np.mean(c_aucs):.4f} ± {np.std(c_aucs):.4f}    "
          f"{np.mean(gaps):+.4f} ± {np.std(gaps):.4f}    "
          f"{winner}")

# Save master JSON
master = {
    'experiment':   'subset_crf_v3',
    'model_a':      'Baseline',
    'model_b':      'CRF (symmetric, 3 mean-field iterations)',
    'architecture': {
        'hidden_dim':      16,
        'proj_dim':        8,
        'n_hidden_layers': 3,
        'crf_n_iter':      3,
        'crf_symmetric':   True,
    },
    'seeds':     SEEDS,
    'fractions': SUBSET_FRACTIONS,
    'results':   {str(s): {str(f): v for f, v in sv.items()}
                  for s, sv in all_results.items()}
}

master_path = os.path.join(BASE_DIR, 'master_results.json')
with open(master_path, 'w') as f:
    json.dump(master, f, indent=2)

shutil.copy2(master_path, os.path.join(DRIVE_DIR, 'master_results.json'))
print(f"\n✅ Master results saved to Drive.")
print(f"{'='*70}")

Mounted at /content/drive

SUBSET EXPERIMENT — CRF INTERACTION (SYMMETRIC, 3 ITERATIONS)
Seeds:     [42, 123, 456, 789, 1337, 2024, 9999]
Fractions: [1.0, 0.8, 0.6, 0.4, 0.2]
Backbone:  hidden_dim=16, proj_dim=8, 3 hidden layers (fixed)
Models:    Baseline vs CRF

SEED: 42

SEED 42 — SUBSET CONDITION: 100% Data
  Using full training set: 2148 samples


Baseline | seed=42, 100% data:   0%|                              | 0/300 [00:00<?, ?epoch/s]

Baseline Test AUROC: 0.6573


CRF | seed=42, 100% data:   0%|                              | 0/300 [00:00<?, ?epoch/s]

CRF Test AUROC: 0.6883
Gap: +0.0310
Beta (co-occurrence weights):
  NEP-NEU: -0.0157
  NEP-RET: -0.0534
  NEU-RET: -0.0269
  ✅ Plot 1: Training dynamics
  ✅ Plot 2: Metric comparison


...

Baseline Test AUROC: 0.5344


CRF | seed=9999, 20% data:   0%|                              | 0/300 [00:00<?, ?epoch/s]

CRF Test AUROC: 0.5716
Gap: +0.0372
Beta (co-occurrence weights):
  NEP-NEU: +0.0624
  NEP-RET: -0.1795
  NEU-RET: +0.1515
  ✅ Plot 1: Training dynamics
  ✅ Plot 2: Metric comparison


/tmp/ipykernel_11306/966174437.py:186: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


  ✅ Plot 3: Per-label AUROC + ROC curves
  ✅ Plot 4: Interaction matrix
  ✅ Model weights and beta matrix saved
  ✅ Results JSON saved
  ✅ Artifacts copied to Drive: /content/drive/MyDrive/comorbidity_experiments/subset_crf_v3/seed_9999/20pct

✅ COMPLETE: seed=9999, 20% data

SUBSET EXPERIMENT — CRF — MASTER SUMMARY

Fraction     Seed     Base AUROC     CRF AUROC        Δ          Winner
----------------------------------------------------------------------
100          42       0.6573         0.6883           +0.0310    CRF ✅
80           42       0.5871         0.6240           +0.0370    CRF ✅
60           42       0.6030         0.6210           +0.0180    CRF ✅
40           42       0.5690         0.5597           -0.0093    Baseline
20           42       0.5459         0.5472           +0.0013    CRF ✅
100          123      0.6540         0.6359           -0.0181    Baseline
80           123      0.6171         0.6311           +0.0140    CRF ✅
60           123      0.5794       